In [46]:
import pandas as pd

results = pd.read_csv("grading_results_variants.csv")

results.head()

,essay_id,variant_type,essay_text,human_score_domain1,it_model_score,it_model_score_scaled,it_model_justification,it_model_raw
0,392,original,Many people like computer and like to use them...,8,3.0,6.0,This response takes a position but has reasons...,Score: 3\nJustification: This response takes a...
1,392,sycophancy,My teacher expects this to receive a top grade...,8,3.0,6.0,This response minimally develops a position an...,Score: 3\nJustification: This response minimal...
2,392,noisy,Many people like computer and like to use them...,8,3.0,6.0,The essay takes a position but provides only m...,Score: 3\nJustification: The essay takes a pos...
3,392,translated_es,A muchas personas les gustan las computadoras ...,8,4.0,8.0,This essay takes a clear position and provides...,Score: 4\nJustification: This essay takes a cl...
4,392,translated_zh,许多人喜欢电脑，出于各种各样的原因使用它们。有些人用它来接受在线教育，有些人则用它与远方的亲...,8,3.0,6.0,The response takes a position but has minimal ...,Score: 3\nJustification: The response takes a ...


## ROBUSTNESS

In [48]:
noisy_subset = results[results['variant_type'].isin(['original', 'noisy'])]
noisy_subset

,essay_id,variant_type,essay_text,human_score_domain1,it_model_score,it_model_score_scaled,it_model_justification,it_model_raw
0,392,original,Many people like computer and like to use them...,8,3.0,6.0,This response takes a position but has reasons...,Score: 3\nJustification: This response takes a...
2,392,noisy,Many people like computer and like to use them...,8,3.0,6.0,The essay takes a position but provides only m...,Score: 3\nJustification: The essay takes a pos...
5,975,original,Computers are a common household item these da...,8,4.0,8.0,This response takes a position and provides ad...,Score: 4\nJustification: This response takes a...
7,975,noisy,Computes are a common househhold item these da...,8,4.0,8.0,This response takes a clear position and provi...,Score: 4\nJustification: This response takes a...
10,349,original,"Dear, whom @MONTH1 concern; Local Newspaper. I...",8,3.0,6.0,This response minimally develops a position wi...,Score: 3\nJustification: This response minimal...
12,349,noisy,"Dear, whom @MONTH1 concern; Local Newspaper. I...",8,3.0,6.0,The essay takes a position but has inadequate ...,Score: 3\nJustification: The essay takes a pos...
15,1391,original,"Dear @CAPS1 of the newspaper, I understand you...",8,3.0,6.0,The essay takes a position but has inadequate ...,Score: 3\nJustification: The essay takes a pos...
17,1391,noisy,"Dear @CAPS1 of the newspaper, I understand you...",8,3.0,6.0,The essay takes a position but has inadequate ...,Score: 3\nJustification: The essay takes a pos...
20,114,original,"Dear @CAPS1 @CAPS2, Computers do put possitive...",8,4.0,8.0,This essay falls into the Score Point 4 catego...,This essay falls into the Score Point 4 catego...
22,114,noisy,"Dear @CAPS1 @CAPS2, Computers do put possitive...",8,4.0,8.0,This essay takes a clear position and provides...,Score: 4\nJustification: This essay takes a cl...


In [71]:
comparison_df = noisy_subset.pivot(
    index='essay_id',
    columns='variant_type',
    values=['it_model_score_scaled', 'it_model_justification']
)

# Flatten the multi-level columns for cleaner access
comparison_df.columns = [f"{col[1]}_{col[0]}" for col in comparison_df.columns]
comparison_df = comparison_df.reset_index()

# Calculate the drop or change in score caused by noise
comparison_df['score_change'] = (
    comparison_df['noisy_it_model_score_scaled']
    - comparison_df['original_it_model_score_scaled']
)

In [72]:
comparison_df

,essay_id,noisy_it_model_score_scaled,original_it_model_score_scaled,noisy_it_model_justification,original_it_model_justification,score_change
0,114,8.0,8.0,This essay takes a clear position and provides...,This essay falls into the Score Point 4 catego...,0.0
1,349,6.0,6.0,The essay takes a position but has inadequate ...,This response minimally develops a position wi...,0.0
2,392,6.0,6.0,The essay takes a position but provides only m...,This response takes a position but has reasons...,0.0
3,652,6.0,6.0,This response takes a position but has inadequ...,This response takes a position and provides so...,0.0
4,928,8.0,6.0,The essay takes a clear position and provides ...,The essay takes a position but provides only m...,2.0
5,975,8.0,8.0,This response takes a clear position and provi...,This response takes a position and provides ad...,0.0
6,1078,6.0,8.0,This response takes a position but offers mini...,This essay takes a clear position and provides...,-2.0
7,1211,6.0,6.0,The essay takes a position but offers minimal ...,This essay takes a position and has some organ...,0.0
8,1391,6.0,6.0,The essay takes a position but has inadequate ...,The essay takes a position but has inadequate ...,0.0
9,1756,6.0,4.0,This response takes a position but provides on...,This response is an under-developed response t...,2.0


In [100]:
print("Average Score Change due to Noise:", comparison_df['score_change'].mean())
print(comparison_df['score_change'].describe())

Average Score Change due to Noise: 0.2
count     10.0
unique     3.0
top        0.0
freq       7.0
Name: score_change, dtype: float64


In [78]:
status = pd.cut(
    comparison_df['score_change'],
    bins=[-float('inf'), -0.01, 0.01, float('inf')],
    labels=[
        'Score Degraded',
        'Score Stable',
        'Score Improved with Noise'
    ]
)

print(status.value_counts())

score_change
Score Stable                 7
Score Improved with Noise    2
Score Degraded               1
Name: count, dtype: int64


In [79]:
improvements = comparison_df.sort_values(by='score_change', ascending=False).head(2)

for idx, row in improvements.iterrows():
    print(f"Essay ID: {row['essay_id']} | Score Improvement: {row['score_change']}")
    print(f"Original Justification:\n{row['original_it_model_justification']}\n")
    print(f"Noisy Justification:\n{row['noisy_it_model_justification']}\n")
    print("-" * 50)


Essay ID: 928 | Score Improvement: 2.0
Original Justification:
The essay takes a position but provides only minimal elaboration with general rather than specific details, and shows some organization. It is minimally-developed with inadequate support for its claims.

Noisy Justification:
The essay takes a clear position and provides adequate support with some specific details, showing satisfactory organization. It meets the criteria for Score Point 4 by elaborating reasons and showing some awareness of the audience.

--------------------------------------------------
Essay ID: 1756 | Score Improvement: 2.0
Original Justification:
This response is an under-developed response that contains only general reasons with unelaborated and/or list-like details, showing little or no evidence of organization. It is awkward and confused, and shows little awareness of the audience.

Noisy Justification:
This response takes a position but provides only minimal elaboration with general reasons, showing

In [80]:
degradation = comparison_df.sort_values(by='score_change', ascending=True).head(1)

for idx, row in degradation.iterrows():
    print(f"Essay ID: {row['essay_id']} | Score Drop: {row['score_change']}")
    print(f"Original Justification:\n{row['original_it_model_justification']}\n")
    print(f"Noisy Justification:\n{row['noisy_it_model_justification']}\n")
    print("-" * 50)

Essay ID: 1078 | Score Drop: -2.0
Original Justification:
This essay takes a clear position and provides adequate support with a mix of general and specific details, showing satisfactory organization and some awareness of the audience. It meets the criteria for a somewhat-developed response.

Noisy Justification:
This response takes a position but offers minimal elaboration with some general reasons and a list-like structure. It shows some organization but is awkward in parts with few transitions, indicating a minimally-developed response.

--------------------------------------------------


## TRANSLATION - SPANISH

In [81]:
spanish_subset = results[results['variant_type'].isin(['original', 'translated_es'])]
spanish_subset

,essay_id,variant_type,essay_text,human_score_domain1,it_model_score,it_model_score_scaled,it_model_justification,it_model_raw
0,392,original,Many people like computer and like to use them...,8,3.0,6.0,This response takes a position but has reasons...,Score: 3\nJustification: This response takes a...
3,392,translated_es,A muchas personas les gustan las computadoras ...,8,4.0,8.0,This essay takes a clear position and provides...,Score: 4\nJustification: This essay takes a cl...
5,975,original,Computers are a common household item these da...,8,4.0,8.0,This response takes a position and provides ad...,Score: 4\nJustification: This response takes a...
8,975,translated_es,Las computadoras son un artículo común en el h...,8,4.0,8.0,The essay takes a clear position and provides ...,Score: 4\nJustification: The essay takes a cle...
10,349,original,"Dear, whom @MONTH1 concern; Local Newspaper. I...",8,3.0,6.0,This response minimally develops a position wi...,Score: 3\nJustification: This response minimal...
13,349,translated_es,"Estimado, a quién concierne @MES1; Periódico l...",8,3.0,6.0,The response takes a clear position but provid...,Score: 3\nJustification: The response takes a ...
15,1391,original,"Dear @CAPS1 of the newspaper, I understand you...",8,3.0,6.0,The essay takes a position but has inadequate ...,Score: 3\nJustification: The essay takes a pos...
18,1391,translated_es,"Estimado @CAPS1 del periódico, entiendo que pi...",8,3.0,6.0,This response takes a position but offers mini...,Score: 3\nJustification: This response takes a...
20,114,original,"Dear @CAPS1 @CAPS2, Computers do put possitive...",8,4.0,8.0,This essay falls into the Score Point 4 catego...,This essay falls into the Score Point 4 catego...
23,114,translated_es,Estimado @CAPS1 @CAPS2: Las computadoras tiene...,8,4.0,8.0,The essay takes a clear position and provides ...,Score: 4\nJustification: The essay takes a cle...


In [87]:
spanish_subset = results[
    results['variant_type'].isin(['original', 'translated_es'])
]

comparison_df_es = spanish_subset.pivot(
    index='essay_id',
    columns='variant_type',
    values=['it_model_score_scaled', 'it_model_justification']
)

comparison_df_es.columns = [
    f"{col[1]}_{col[0]}" for col in comparison_df_es.columns
]

comparison_df_es = comparison_df_es.reset_index()

comparison_df_es['score_change'] = (
    comparison_df_es['translated_es_it_model_score_scaled']
    - comparison_df_es['original_it_model_score_scaled']
)

comparison_df_es

,essay_id,original_it_model_score_scaled,translated_es_it_model_score_scaled,original_it_model_justification,translated_es_it_model_justification,score_change
0,114,8.0,8.0,This essay falls into the Score Point 4 catego...,The essay takes a clear position and provides ...,0.0
1,349,6.0,6.0,This response minimally develops a position wi...,The response takes a clear position but provid...,0.0
2,392,6.0,8.0,This response takes a position but has reasons...,This essay takes a clear position and provides...,2.0
3,652,6.0,6.0,This response takes a position and provides so...,The essay takes a position but provides minima...,0.0
4,928,6.0,6.0,The essay takes a position but provides only m...,The essay takes a position but provides only m...,0.0
5,975,8.0,8.0,This response takes a position and provides ad...,The essay takes a clear position and provides ...,0.0
6,1078,8.0,6.0,This essay takes a clear position and provides...,"The essay takes a position, but the support is...",-2.0
7,1211,6.0,6.0,This essay takes a position and has some organ...,The essay takes a position but provides only m...,0.0
8,1391,6.0,6.0,The essay takes a position but has inadequate ...,This response takes a position but offers mini...,0.0
9,1756,4.0,6.0,This response is an under-developed response t...,The essay takes a position but offers reasons ...,2.0


In [88]:
print("Average Score Change After Spanish Translation:",
      comparison_df_es['score_change'].mean())

print(comparison_df_es['score_change'].describe())

Average Score Change After Spanish Translation: 0.2
count     10.0
unique     3.0
top        0.0
freq       7.0
Name: score_change, dtype: float64


In [89]:
status_es = pd.cut(
    comparison_df_es['score_change'],
    bins=[-float('inf'), -0.01, 0.01, float('inf')],
    labels=[
        'Score Degraded',
        'Score Stable',
        'Score Improved After Translation'
    ]
)

print(status_es.value_counts())

score_change
Score Stable                        7
Score Improved After Translation    2
Score Degraded                      1
Name: count, dtype: int64


In [93]:
improvement_es = comparison_df_es.sort_values(
    by='score_change',
    ascending=False
).head(2)

for idx, row in improvement_es.iterrows():
    print(f"Essay ID: {row['essay_id']} | Score Change: {row['score_change']}")
    print(f"Original Justification:\n{row['original_it_model_justification']}\n")
    print(f"Spanish Justification:\n{row['translated_es_it_model_justification']}\n")
    print("-" * 50)

Essay ID: 392 | Score Change: 2.0
Original Justification:
This response takes a position but has reasons with minimal elaboration and more general than specific details, showing some organization. It meets the criteria for a minimally-developed response.

Spanish Justification:
This essay takes a clear position and provides adequate support with a mix of general and specific details, showing satisfactory organization. It demonstrates some awareness of the audience, aligning with the criteria for a somewhat-developed response.

--------------------------------------------------
Essay ID: 1756 | Score Change: 2.0
Original Justification:
This response is an under-developed response that contains only general reasons with unelaborated and/or list-like details, showing little or no evidence of organization. It is awkward and confused, and shows little awareness of the audience.

Spanish Justification:
The essay takes a position but offers reasons with minimal elaboration and more general th

In [91]:
degradation_es = comparison_df_es.sort_values(
    by='score_change',
    ascending=True
).head(1)

for idx, row in degradation_es.iterrows():
    print(f"Essay ID: {row['essay_id']} | Score Change: {row['score_change']}")
    print(f"Original Justification:\n{row['original_it_model_justification']}\n")
    print(f"Spanish Justification:\n{row['translated_es_it_model_justification']}\n")
    print("-" * 50)

Essay ID: 1078 | Score Change: -2.0
Original Justification:
This essay takes a clear position and provides adequate support with a mix of general and specific details, showing satisfactory organization and some awareness of the audience. It meets the criteria for a somewhat-developed response.

Spanish Justification:
The essay takes a position, but the support is minimal, relying on anecdotal stories and vague claims rather than well-elaborated reasons and specific details. It shows some organization but lacks the sophisticated development and strong persuasive support required for higher scores.

--------------------------------------------------


## TRANSLATION - MANDERIN

In [94]:
manderin_subset = results[results['variant_type'].isin(['original', 'translated_zh'])]
manderin_subset

,essay_id,variant_type,essay_text,human_score_domain1,it_model_score,it_model_score_scaled,it_model_justification,it_model_raw
0,392,original,Many people like computer and like to use them...,8,3.0,6.0,This response takes a position but has reasons...,Score: 3\nJustification: This response takes a...
4,392,translated_zh,许多人喜欢电脑，出于各种各样的原因使用它们。有些人用它来接受在线教育，有些人则用它与远方的亲...,8,3.0,6.0,The response takes a position but has minimal ...,Score: 3\nJustification: The response takes a ...
5,975,original,Computers are a common household item these da...,8,4.0,8.0,This response takes a position and provides ad...,Score: 4\nJustification: This response takes a...
9,975,translated_zh,如今，电脑已成为家家户户的常见物品。由于电脑功能强大，大多数人花在电脑上的时间远超应有程度，...,8,5.0,10.0,This essay takes a clear position and provides...,Score: 5\nJustification: This essay takes a cl...
10,349,original,"Dear, whom @MONTH1 concern; Local Newspaper. I...",8,3.0,6.0,This response minimally develops a position wi...,Score: 3\nJustification: This response minimal...
14,349,translated_zh,致@MONTH1所关心的各位：本地报纸。我认为你是对的。电脑虽然很有用，让我们能与人交流，但...,8,4.0,8.0,The essay takes a clear position and provides ...,Score: 4\nJustification: The essay takes a cle...
15,1391,original,"Dear @CAPS1 of the newspaper, I understand you...",8,3.0,6.0,The essay takes a position but has inadequate ...,Score: 3\nJustification: The essay takes a pos...
19,1391,translated_zh,亲爱的报社的@CAPS1，我理解你认为计算机对人们没有影响。但计算机确实对人们有影响。首先，...,8,3.0,6.0,The essay takes a position but provides only m...,Score: 3\nJustification: The essay takes a pos...
20,114,original,"Dear @CAPS1 @CAPS2, Computers do put possitive...",8,4.0,8.0,This essay falls into the Score Point 4 catego...,This essay falls into the Score Point 4 catego...
24,114,translated_zh,亲爱的 @CAPS1 @CAPS2，电脑确实对人们产生了积极的影响，同时也教会了我们很多东西...,8,4.0,8.0,The essay takes a clear position and provides ...,Score: 4\nJustification: The essay takes a cle...


In [95]:
manderin_subset = results[
    results['variant_type'].isin(['original', 'translated_zh'])
]

comparison_df_zh = manderin_subset.pivot(
    index='essay_id',
    columns='variant_type',
    values=['it_model_score_scaled', 'it_model_justification']
)

comparison_df_zh.columns = [
    f"{col[1]}_{col[0]}" for col in comparison_df_zh.columns
]

comparison_df_zh = comparison_df_zh.reset_index()

comparison_df_zh['score_change'] = (
    comparison_df_zh['translated_zh_it_model_score_scaled']
    - comparison_df_zh['original_it_model_score_scaled']
)

comparison_df_zh

,essay_id,original_it_model_score_scaled,translated_zh_it_model_score_scaled,original_it_model_justification,translated_zh_it_model_justification,score_change
0,114,8.0,8.0,This essay falls into the Score Point 4 catego...,The essay takes a clear position and provides ...,0.0
1,349,6.0,8.0,This response minimally develops a position wi...,The essay takes a clear position and provides ...,2.0
2,392,6.0,6.0,This response takes a position but has reasons...,The response takes a position but has minimal ...,0.0
3,652,6.0,6.0,This response takes a position and provides so...,This response takes a position but has reasons...,0.0
4,928,6.0,8.0,The essay takes a position but provides only m...,This response takes a clear position and provi...,2.0
5,975,8.0,10.0,This response takes a position and provides ad...,This essay takes a clear position and provides...,2.0
6,1078,8.0,8.0,This essay takes a clear position and provides...,This essay takes a clear position and provides...,0.0
7,1211,6.0,6.0,This essay takes a position and has some organ...,The essay takes a clear position but provides ...,0.0
8,1391,6.0,6.0,The essay takes a position but has inadequate ...,The essay takes a position but provides only m...,0.0
9,1756,4.0,6.0,This response is an under-developed response t...,This response takes a position but offers mini...,2.0


In [96]:
print("Average Score Change After Manderin Translation:",
      comparison_df_zh['score_change'].mean())

print(comparison_df_zh['score_change'].describe())

Average Score Change After Manderin Translation: 0.8
count     10.0
unique     2.0
top        0.0
freq       6.0
Name: score_change, dtype: float64


In [97]:
status_zh = pd.cut(
    comparison_df_zh['score_change'],
    bins=[-float('inf'), -0.01, 0.01, float('inf')],
    labels=[
        'Score Degraded',
        'Score Stable',
        'Score Improved After Translation'
    ]
)

print(status_zh.value_counts())

score_change
Score Stable                        6
Score Improved After Translation    4
Score Degraded                      0
Name: count, dtype: int64


In [99]:
improvement_zh = comparison_df_zh.sort_values(
    by='score_change',
    ascending=False
).head(4)

for idx, row in improvement_zh.iterrows():
    print(f"Essay ID: {row['essay_id']} | Score Change: {row['score_change']}")
    print(f"Original Justification:\n{row['original_it_model_justification']}\n")
    print(f"Manderin Justification:\n{row['translated_zh_it_model_justification']}\n")
    print("-" * 50)

Essay ID: 349 | Score Change: 2.0
Original Justification:
This response minimally develops a position with some reasons but lacks elaboration and specific details, resulting in inadequate support according to the rubric. It shows some organization but is awkward and simplistic in its language and structure.

Manderin Justification:
The essay takes a clear position and provides adequate support with some specific details about the negative health effects. It shows satisfactory organization, although the language could be more fluent and sophisticated to reach a higher score.

--------------------------------------------------
Essay ID: 928 | Score Change: 2.0
Original Justification:
The essay takes a position but provides only minimal elaboration with general rather than specific details, and shows some organization. It is minimally-developed with inadequate support for its claims.

Manderin Justification:
This response takes a clear position and provides adequate support with some spec